# Generate paper diagrams as PNG (for Word)

Redraws the three TikZ diagrams of `ijece-en2.tex` using **matplotlib only** (no LaTeX / TikZ needed), and saves high-resolution PNGs ready to Insert into the IJECE Word template:

- **fig1.png** - Proposed system architecture (Figure 1)
- **fig2.png** - AES-256 encryption/decryption (Figure 2)
- **fig4.png** - VHDX air-gapped backup architecture (Figure 4)

> Naming follows the figure NUMBER in the paper. Figure 3 (compression chart) is
> an existing PNG, so there is no fig3.png here.

> Figure 3 (compression chart) is already a PNG (`figures/figure10_compression_en.png`) and does not need regeneration.

**Requirement:** only `matplotlib`. Install if needed:
```
pip install matplotlib
```
Output PNGs are written to `figures_png/` at 300 DPI.

In [ ]:
import os
import matplotlib
matplotlib.use('Agg')  # no display needed; just save files
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

OUTDIR = 'figures_png'
os.makedirs(OUTDIR, exist_ok=True)
DPI = 300

# Colors approximating the TikZ fills (blue!6, red!8, orange!10, green!8, blue!15 ...)
C_BLUE_LT   = '#eaf1fb'
C_BLUE_MD   = '#c5d9f1'
C_RED_LT    = '#fbe9e7'
C_ORANGE_LT = '#fdecd9'
C_GREEN_LT  = '#e6f3e6'
C_WHITE     = '#ffffff'
EDGE        = '#333333'
ARROW_BLUE  = '#2b4a6f'
ARROW_DARK  = '#222222'

def box(ax, x, y, w, h, text, fc=C_BLUE_LT, ec=EDGE, fs=10, rounded=True, lw=1.3):
    """Draw a rounded box centered at (x,y). Returns (x,y,w,h)."""
    style = 'round,pad=0.02,rounding_size=0.12' if rounded else 'square,pad=0.02'
    p = FancyBboxPatch((x - w/2, y - h/2), w, h, boxstyle=style,
                       facecolor=fc, edgecolor=ec, linewidth=lw, zorder=2)
    ax.add_patch(p)
    ax.text(x, y, text, ha='center', va='center', fontsize=fs, zorder=3)
    return (x, y, w, h)

def arrow(ax, p1, p2, color=ARROW_BLUE, lw=1.6, label=None, label_fs=8, dashed=False, label_dy=0.12):
    ls = (0,(5,3)) if dashed else '-'
    a = FancyArrowPatch(p1, p2, arrowstyle='-|>', mutation_scale=14,
                        color=color, linewidth=lw, linestyle=ls, zorder=1,
                        shrinkA=2, shrinkB=2)
    ax.add_patch(a)
    if label:
        mx, my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
        ax.text(mx, my + label_dy, label, ha='center', va='bottom', fontsize=label_fs, zorder=3)

def finish(ax, xlim, ylim):
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect('equal'); ax.axis('off')

print('helpers ready')

## Figure 1 - Proposed system architecture
Top row (left->right): User -> Production Data -> Ransomware Detection -> AES-256 Encryption -> Compression (5 algorithms) -> Ransomware Attack.
Then down to Retrieve Backup (Air-Gapped), and bottom row (right->left): Recovery -> AES-256 Decryption -> Data Validation (SHA-256) -> Evaluation.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.2))
w, h = 1.9, 1.0
dx = 2.25
y_top = 3.0
y_bot = 0.6

# top row
xs = [0 + i*dx for i in range(6)]
box(ax, xs[0], y_top, w, h, 'User\n(data access)', fc=C_BLUE_LT)
box(ax, xs[1], y_top, w, h, 'Production\nData', fc=C_BLUE_LT)
box(ax, xs[2], y_top, w, h, 'Ransomware\nDetection', fc=C_BLUE_LT)
box(ax, xs[3], y_top, w, h, 'AES-256\nEncryption', fc=C_BLUE_LT)
box(ax, xs[4], y_top, w, h, 'Compression\n(5 algorithms)', fc=C_BLUE_LT)
box(ax, xs[5], y_top, w, h, 'Ransomware\nAttack', fc=C_RED_LT)
for i in range(5):
    arrow(ax, (xs[i]+w/2, y_top), (xs[i+1]-w/2, y_top))

# down to backup (under the attack box)
box(ax, xs[5], y_bot, w, h, 'Retrieve Backup\n(Air-Gapped)', fc=C_ORANGE_LT)
arrow(ax, (xs[5], y_top-h/2), (xs[5], y_bot+h/2))

# bottom row right->left
box(ax, xs[4], y_bot, w, h, 'Recovery\n(Restore)', fc=C_BLUE_LT)
box(ax, xs[3], y_bot, w, h, 'AES-256\nDecryption', fc=C_BLUE_LT)
box(ax, xs[2], y_bot, w, h, 'Data Validation\n(SHA-256)', fc=C_BLUE_LT)
box(ax, xs[1], y_bot, w, h, 'Evaluation', fc=C_GREEN_LT)
for a, b in [(5,4),(4,3),(3,2),(2,1)]:
    arrow(ax, (xs[a]-w/2, y_bot), (xs[b]+w/2, y_bot))

finish(ax, (xs[0]-w/2-0.3, xs[5]+w/2+0.3), (y_bot-h/2-0.3, y_top+h/2+0.3))
plt.tight_layout()
out = os.path.join(OUTDIR, 'fig1.png')
fig.savefig(out, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close(fig)
print('saved', out)

## Figure 2 - AES-256 encryption and decryption (symmetric key)
Sender -> Encryption Server -> (Ciphertext) -> Decryption Server -> Receiver; both servers receive the Secret Key over a Secure Channel.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.0))
w, h = 1.8, 0.95
y = 1.0
x_sender, x_enc, x_dec, x_recv = 0, 2.6, 6.0, 8.6

box(ax, x_sender, y, w, h, 'Sender', fc=C_WHITE)
box(ax, x_enc,    y, w, h, 'Encryption\nServer', fc=C_BLUE_MD)
box(ax, x_dec,    y, w, h, 'Decryption\nServer', fc=C_BLUE_MD)
box(ax, x_recv,   y, w, h, 'Receiver', fc=C_WHITE)

# keys above the two servers
yk = y + 2.0
box(ax, x_enc, yk, 1.6, 0.7, 'Secret Key', fc=C_BLUE_LT, fs=9)
box(ax, x_dec, yk, 1.6, 0.7, 'Secret Key', fc=C_BLUE_LT, fs=9)

arrow(ax, (x_sender+w/2, y), (x_enc-w/2, y), label='Plaintext')
arrow(ax, (x_enc+w/2, y), (x_dec-w/2, y), label='Ciphertext', lw=2.2)
arrow(ax, (x_dec+w/2, y), (x_recv-w/2, y), label='Plaintext')
arrow(ax, (x_enc, yk-0.7/2), (x_enc, y+h/2))
arrow(ax, (x_dec, yk-0.7/2), (x_dec, y+h/2))
arrow(ax, (x_enc+1.6/2, yk), (x_dec-1.6/2, yk), label='Secure Channel', color=ARROW_DARK)

finish(ax, (x_sender-w/2-0.3, x_recv+w/2+0.3), (y-h/2-0.4, yk+0.7/2+0.3))
plt.tight_layout()
out = os.path.join(OUTDIR, 'fig2.png')
fig.savefig(out, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close(fig)
print('saved', out)

## Figure 4 - VHDX media-based air-gapped backup architecture
Main System connects to the Primary Repository under normal conditions; on a Ransomware Indication the restore path switches to the Air-Gapped Repository (mounted only during backup/restore, isolated from the network).

In [ ]:
from matplotlib.patches import Ellipse, Rectangle

def cylinder(ax, x, y, w, h, text, fc=C_GREEN_LT, ec=EDGE, fs=9, lw=1.3):
    """Simple database-cylinder shape centered at (x,y)."""
    ell_h = h*0.22
    body = Rectangle((x-w/2, y-h/2+ell_h/2), w, h-ell_h, facecolor=fc, edgecolor='none', zorder=2)
    ax.add_patch(body)
    bot = Ellipse((x, y-h/2+ell_h/2), w, ell_h, facecolor=fc, edgecolor=ec, linewidth=lw, zorder=2)
    top = Ellipse((x, y+h/2-ell_h/2), w, ell_h, facecolor=fc, edgecolor=ec, linewidth=lw, zorder=3)
    ax.add_patch(bot); ax.add_patch(top)
    ax.plot([x-w/2, x-w/2], [y-h/2+ell_h/2, y+h/2-ell_h/2], color=ec, lw=lw, zorder=2)
    ax.plot([x+w/2, x+w/2], [y-h/2+ell_h/2, y+h/2-ell_h/2], color=ec, lw=lw, zorder=2)
    ax.text(x, y, text, ha='center', va='center', fontsize=fs, zorder=4)

fig, ax = plt.subplots(figsize=(10, 5.0))

x_att, x_sys, x_rep = 0.0, 3.2, 6.6
y_top = 3.0
y_air = 0.6

box(ax, x_att, y_top, 2.4, 1.0, 'Ransomware\nIndication', fc=C_RED_LT)
box(ax, x_sys, y_top, 2.6, 1.0, 'Main System\n(Backup/Restore)', fc=C_BLUE_LT)
cylinder(ax, x_rep, y_top, 1.8, 1.5, 'Primary\nRepository', fc=C_GREEN_LT)
cylinder(ax, x_rep, y_air, 1.8, 1.5, 'Air-Gapped\nRepository\n(Disk D:/G:)', fc=C_ORANGE_LT)

arrow(ax, (x_sys+2.6/2, y_top), (x_rep-1.8/2, y_top), label='normal')
arrow(ax, (x_att+2.4/2, y_top), (x_sys-2.6/2, y_top), dashed=True)
# system down to air-gapped (elbow)
a = FancyArrowPatch((x_sys, y_top-1.0/2), (x_rep-1.8/2, y_air), arrowstyle='-|>',
                    mutation_scale=14, color=ARROW_BLUE, linewidth=1.6,
                    connectionstyle='angle,angleA=-90,angleB=180,rad=0', zorder=1)
ax.add_patch(a)
ax.text(x_sys+0.3, y_air-0.1, 'during attack', ha='left', va='center', fontsize=8)
ax.text(x_rep, y_air-1.1, 'mounted only during backup/restore\n(isolated from the network)',
        ha='center', va='top', fontsize=8, style='italic')

finish(ax, (x_att-2.4/2-0.3, x_rep+1.8/2+0.3), (y_air-1.9, y_top+1.0))
plt.tight_layout()
out = os.path.join(OUTDIR, 'fig4.png')
fig.savefig(out, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close(fig)
print('saved', out)

In [ ]:
# list the generated PNGs
for f in sorted(os.listdir(OUTDIR)):
    p = os.path.join(OUTDIR, f)
    print(f, '-', os.path.getsize(p), 'bytes')
print('\nDone. Insert these PNGs into Word via Insert > Pictures.')